# 11 — Thermal-sensitivity falsification test (Paso 3, user protocol 2026-09-11)

Re-runs the actual O3 equation discovery on the same selected no-lag feature set
(`photochemical_potential, no, pm25, rh_2m, w700, day_of_week, no2, precip_total_mm`,
the config that gives LOYO R2=0.640 in the main pipeline) but adds `temp_2m` as an
additional candidate variable in the PySR search pool.

- If PySR replaces the `pm25`/`photochemical_potential`-driven interaction with a
  `temp_2m*radiation`-shaped term, PM2.5 was a statistical stand-in for thermal energy
  (matching the temperature-tercile result showing the interaction strengthens on warm
  days -- Hipotesis B/C, evaporative-VOC or direct-thermal pathways).
- If PySR keeps a real PM2.5-involving term ALONGSIDE a temp_2m term on the Pareto
  front, secondary particulate mass and temperature exert independent predictive
  pressure on O3.


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)

with open(DATA_DIR / "bogota_o3_feature_selection.json") as f:
    selection = json.load(f)
base_features = selection['selected_no_lag']
features = base_features + ['temp_2m']
print("Base (existing R2=0.640) features:", base_features)
print("Features with temp_2m added:", features)

data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy()
y = data['o3'].to_numpy()
print(data.shape)


Base (existing R2=0.640) features: ['photochemical_potential', 'hotspots_sector_E_lag2', 'fog_stagnation', 'wind_speed_era5', 'no_ratio', 'w700', 'radiation_mean']
Features with temp_2m added: ['photochemical_potential', 'hotspots_sector_E_lag2', 'fog_stagnation', 'wind_speed_era5', 'no_ratio', 'w700', 'radiation_mean', 'temp_2m']
(2917, 219)


## PySR with temp_2m as a candidate alongside the existing selected features

In [2]:
from pysr import PySRRegressor

sr = PySRRegressor(
    niterations=200,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "sqrt", "exp"],
    model_selection="best",
    maxsize=25,
    populations=24,
)
sr.fit(X, y, variable_names=features)
print(sr)


Compiling Julia backend...


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 9.240e+03
Progress: 22 / 200 total iterations (11.229%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.894e+01  0.000e+00  y = 20.162
3           4.394e+01  5.395e-02  y = no_ratio + 18.604
4           3.691e+01  1.742e-01  y = sqrt(photochemical_potential * 0.10212)
5           3.674e+01  4.699e-03  y = sqrt(sqrt(photochemical_potential)) / 0.38863
6           2.839e+01  2.578e-01  y = sqrt(radiation_mean + (no_ratio * radiation_mean))
7           2.828e+01  3.950e-03  y = sqrt((radiation_mean + radiation_mean) * sqrt(no_ratio...
                                      ))
8           2.614e+01  7.878e-02  y = sqrt(radiation_mean + (radiation_mean * (no_ratio - w7...
                                      00)))
9           2.242e+01  1.535e-01  y 

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          20.142193   
	1         0.083494                       radiation_mean * 0.105226286   
	2         0.115159        sqrt(photochemical_potential * 0.100972876)   
	3         0.031826          no_ratio + (radiation_mean * 0.098778084)   
	4         0.230555  sqrt(radiation_mean + (no_ratio * radiation_me...   
	5         0.013289  sqrt(radiation_mean * sqrt(4.4190683 * no_ratio))   
	6         0.086689  sqrt((radiation_mean / wind_speed_era5) + (no_...   
	7   >>>>  0.136215  sqrt(sqrt(hotspots_sector_E_lag2 * radiation_m...   
	8         0.003233  sqrt(sqrt(no_ratio) * (radiation_mean + sqrt(r...   
	9         0.040172  sqrt(sqrt((hotspots_sector_E_lag2 / wind_speed...   
	10        0.012127  sqrt(radiation_mean * sqrt(sqrt((hotspots_sect...   
	11        0.054579  sqrt(radiation_mean * sqrt(no_ratio * ((sqrt(s...   
	12      

In [3]:
pareto = sr.equations_[['complexity', 'loss', 'equation']]
pareto['r2'] = pareto.apply(lambda row: r2_score(y, sr.predict(X, index=row.name)), axis=1)
pareto


,complexity,loss,equation,r2
0,1,48.941936,20.142193,-1.982229e-07
1,3,41.415170,radiation_mean * 0.105226286,1.537895e-01
2,4,36.910210,sqrt(photochemical_potential * 0.100972876),2.458366e-01
3,5,35.754005,no_ratio + (radiation_mean * 0.098778084),2.694605e-01
4,6,28.391994,sqrt(radiation_mean + (no_ratio * radiation_me...,4.198840e-01
5,7,28.017178,sqrt(radiation_mean * sqrt(4.4190683 * no_ratio)),4.275424e-01
6,8,25.690702,sqrt((radiation_mean / wind_speed_era5) + (no_...,4.750778e-01
7,9,22.419127,sqrt(sqrt(hotspots_sector_E_lag2 * radiation_m...,5.419239e-01
8,10,22.346758,sqrt(sqrt(no_ratio) * (radiation_mean + sqrt(r...,5.434025e-01
9,11,21.466843,sqrt(sqrt((hotspots_sector_E_lag2 / wind_speed...,5.613813e-01


## Interpretation: does temp_2m displace pm25, or coexist with it?

Check the highest-R2 equations for whether `pm25` and `temp_2m` both appear, only one
does, or neither in favor of some other combination.


In [4]:
print("Full Pareto front:")
for _, row in pareto.sort_values('complexity').iterrows():
    has_pm25 = 'pm25' in row['equation']
    has_temp = 'temp_2m' in row['equation']
    print(f"  [{int(row['complexity']):2d}] R2={row['r2']:.3f}  pm25={has_pm25}  temp_2m={has_temp}  {row['equation']}")

print()
print("Best (highest-R2) equation:")
best = pareto.loc[pareto['r2'].idxmax()]
print(best['equation'])
print(f"R2 = {best['r2']:.3f}  (vs. existing no-temp_2m best in-sample R2, for comparison)")


Full Pareto front:
  [ 1] R2=-0.000  pm25=False  temp_2m=False  20.142193
  [ 3] R2=0.154  pm25=False  temp_2m=False  radiation_mean * 0.105226286
  [ 4] R2=0.246  pm25=False  temp_2m=False  sqrt(photochemical_potential * 0.100972876)
  [ 5] R2=0.269  pm25=False  temp_2m=False  no_ratio + (radiation_mean * 0.098778084)
  [ 6] R2=0.420  pm25=False  temp_2m=False  sqrt(radiation_mean + (no_ratio * radiation_mean))
  [ 7] R2=0.428  pm25=False  temp_2m=False  sqrt(radiation_mean * sqrt(4.4190683 * no_ratio))
  [ 8] R2=0.475  pm25=False  temp_2m=False  sqrt((radiation_mean / wind_speed_era5) + (no_ratio * radiation_mean))
  [ 9] R2=0.542  pm25=False  temp_2m=False  sqrt(sqrt(hotspots_sector_E_lag2 * radiation_mean) + (radiation_mean * no_ratio))
  [10] R2=0.543  pm25=False  temp_2m=False  sqrt(sqrt(no_ratio) * (radiation_mean + sqrt(radiation_mean * hotspots_sector_E_lag2)))
  [11] R2=0.561  pm25=False  temp_2m=False  sqrt(sqrt((hotspots_sector_E_lag2 / wind_speed_era5) * radiation_mean) + 

## Save

In [5]:
pareto.to_csv(DATA_DIR / "bogota_o3_temp_falsification_pareto.csv", index=False)
print("Saved")


Saved
